# Text Mining Final Project - Group 60

In this project we solve three NLP tasks on the test set: Named Entity Recognition,
sentiment analysis and topic classification. For each task we try two methods from the labs
and compare them. The main comparison and error analysis is on the NER task.

- NER: spaCy and NLTK (Lab 1)
- Sentiment: VADER and Naive Bayes (Lab 3)
- Topic: a keyword method and LDA (Lab 6)

### Setup

In [ ]:

import sys
!{sys.executable} -m pip install vaderSentiment gensim
!{sys.executable} -m spacy download en_core_web_sm

In [ ]:
import json
import pandas as pd

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

import spacy
from spacy.tokens import Doc

import nltk
from nltk import pos_tag, ne_chunk
from nltk.chunk import tree2conlltags

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# nltk data needed for tokenising, POS tagging and the NER chunker
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')

### Load the data

In [ ]:

ner = pd.read_csv("NER-test.tsv", sep="\t")
sent = pd.read_csv("Sentiment-topic-test.tsv", sep="\t")


ner.columns = [c.strip() for c in ner.columns]
sent.columns = [c.strip() for c in sent.columns]


with open("my_tweets.json") as f:
    my_tweets = json.load(f)

print("NER tokens:", len(ner))
print("test sentences:", len(sent))
print("training tweets:", len(my_tweets))
print()
print("sentiment in test set:")
print(sent["sentiment"].value_counts())
print()
print("topics in test set:")
print(sent["topic"].value_counts())

The test set has 10 sentences. For NER they are annotated token by token with the BIO scheme
(PER, ORG, LOC, MISC). The sentences are reviews about movies, restaurants and books. The topic
classes are not balanced (5 movie, 3 restaurant, 2 book). For Naive Bayes we use our 50 self-made
tweets from Lab 3 (17 positive, 17 negative, 16 neutral).

## Task 1: Named Entity Recognition

We compare spaCy and NLTK. We run both on the test sentences and compare the entities they find
to the gold labels. spaCy and NLTK use different label names, so first we map them to the four
labels of the test set (PER, ORG, LOC, MISC).

In [ ]:
# put the gold tokens and tags of each sentence in a list
sentences = []
for sid in sorted(ner["sentence id"].unique()):
    rows = ner[ner["sentence id"] == sid]
    tokens = list(rows["token"])
    tags = list(rows["BIO NER tag"])
    sentences.append((tokens, tags))

# map the spaCy / nltk labels to the labels used in the test set
spacy_map = {"PERSON": "PER", "ORG": "ORG", "GPE": "LOC", "LOC": "LOC",
             "NORP": "MISC", "LANGUAGE": "MISC"}
nltk_map = {"PERSON": "PER", "ORGANIZATION": "ORG", "GPE": "LOC", "LOCATION": "LOC"}

In [ ]:
nlp = spacy.load("en_core_web_sm")

def spacy_tags(tokens):
    # we give spaCy the gold tokens so the tags line up with the gold tags
    doc = Doc(nlp.vocab, words=tokens)
    for name, component in nlp.pipeline:
        doc = component(doc)
    tags = []
    for token in doc:
        if token.ent_iob_ == "O" or token.ent_type_ not in spacy_map:
            tags.append("O")
        else:
            tags.append(token.ent_iob_ + "-" + spacy_map[token.ent_type_])
    return tags

def nltk_tags(tokens):
    chunks = tree2conlltags(ne_chunk(pos_tag(tokens)))
    tags = []
    for word, pos, label in chunks:
        if label == "O":
            tags.append("O")
        else:
            prefix = label[:2]          # "B-" or "I-"
            ne_type = label[2:]
            tags.append(prefix + nltk_map.get(ne_type, "MISC"))
    return tags

In [ ]:
# run both taggers on every sentence and collect all the tags in one big list
gold_all, spacy_all, nltk_all = [], [], []
for tokens, tags in sentences:
    gold_all += tags
    spacy_all += spacy_tags(tokens)
    nltk_all += nltk_tags(tokens)

labels = ["B-PER","I-PER","B-ORG","I-ORG","B-LOC","I-LOC","B-MISC","I-MISC"]

print("spaCy")
print(classification_report(gold_all, spacy_all, labels=labels, zero_division=0))
print("NLTK")
print(classification_report(gold_all, nltk_all, labels=labels, zero_division=0))

In [ ]:
# also check how many whole entities each tagger gets exactly right
def get_entities(tokens, tags):
    entities = []
    i = 0
    while i < len(tags):
        if tags[i].startswith("B-"):
            ne_type = tags[i][2:]
            start = i
            i += 1
            while i < len(tags) and tags[i] == "I-" + ne_type:
                i += 1
            entities.append((start, i, ne_type))
        else:
            i += 1
    return entities

def entity_score(tagger):
    correct = predicted = total = 0
    for tokens, tags in sentences:
        gold = set(get_entities(tokens, tags))
        pred = set(get_entities(tokens, tagger(tokens)))
        correct += len(gold & pred)
        predicted += len(pred)
        total += len(gold)
    precision = correct / predicted
    recall = correct / total
    f1 = 2 * precision * recall / (precision + recall)
    return correct, total, round(f1, 2)

print("spaCy entities correct:", entity_score(spacy_tags))
print("NLTK entities correct:", entity_score(nltk_tags))

In [ ]:
# print the entities side by side to see where the mistakes are
for i, (tokens, tags) in enumerate(sentences):
    gold = [(" ".join(tokens[s:e]), t) for s, e, t in get_entities(tokens, tags)]
    sp = [(" ".join(tokens[s:e]), t) for s, e, t in get_entities(tokens, spacy_tags(tokens))]
    nl = [(" ".join(tokens[s:e]), t) for s, e, t in get_entities(tokens, nltk_tags(tokens))]
    print("sentence", i)
    print("  gold :", gold)
    print("  spaCy:", sp)
    print("  nltk :", nl)

### Results and error analysis

spaCy does clearly better than NLTK (entity F1 0.76 against 0.46).

The biggest reason is that NLTK does not have a MISC label. Because of this it gets all the
nationality and language words wrong: *Italian*, *English* and *African American* all become LOC,
which is why NLTK scores 0 on MISC. NLTK also splits some names, for example *Cuba Gooding Jr.*
becomes *Cuba Gooding* and *Chris O'Donnell* becomes only *Chris*.

spaCy's mistakes are smaller. It drops the title in *Dame Maggie Smith* and *Mr. Kruno*, it adds
*the* to *the New York University*, and it labels the Dutch restaurant *Blauwbrug* as MISC instead
of ORG. NLTK actually keeps *Mr. Kruno* together, so the two tools make different mistakes.

Both taggers are trained on news text, so they have a harder time on these review sentences and on
foreign names like *Blauwbrug*. This matches what the lectures said about models losing accuracy on
a new domain. To improve this we could train spaCy on review data, or add a list of nationalities
for NLTK so it can handle MISC.

## Task 2: Sentiment Analysis

We compare VADER (a rule based method) with Naive Bayes (a model we train ourselves). Both predict
positive, negative or neutral for each test sentence.

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    score = analyzer.polarity_scores(text)["compound"]
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

sent["vader"] = sent["text"].apply(vader_sentiment)

order = ["negative", "neutral", "positive"]
print(classification_report(sent["sentiment"], sent["vader"], labels=order, zero_division=0))
print(confusion_matrix(sent["sentiment"], sent["vader"], labels=order))

In [ ]:
# train Naive Bayes on our 50 tweets
train_texts = [t["text_of_tweet"] for t in my_tweets.values()]
train_labels = [t["sentiment_label"] for t in my_tweets.values()]

vectorizer = TfidfVectorizer(min_df=2)
X_train = vectorizer.fit_transform(train_texts)

model = MultinomialNB()
model.fit(X_train, train_labels)

# predict the test sentences
X_test = vectorizer.transform(sent["text"])
sent["nb"] = model.predict(X_test)

print(classification_report(sent["sentiment"], sent["nb"], labels=order, zero_division=0))
print(confusion_matrix(sent["sentiment"], sent["nb"], labels=order))

In [ ]:
# compare both methods per sentence
for i in range(len(sent)):
    print("sentence", i,
          "| gold:", sent["sentiment"][i],
          "| vader:", sent["vader"][i],
          "| nb:", sent["nb"][i])

### Results and error analysis

VADER gets 0.60 accuracy and Naive Bayes 0.50, but they make different kinds of mistakes.

VADER predicts *positive* too often: all four of its mistakes are sentences it thinks are positive
but are not (sentences 2, 4, 5 and 9). This happens with sentences like *"really trendy but they
have forgotten the food"* - VADER sees the positive word *trendy* and the *but* part is not strong
enough to change the score.

Naive Bayes makes more mixed mistakes. It was only trained on 50 short tweets, so most words in the
movie/book/restaurant sentences are new to it. For example it labels *"Blauwbrug has been our
favorite place to eat"* as negative and *"the disaster that was this movie"* as positive. So VADER
is biased towards positive, while Naive Bayes is more random because it has too little training data.

To improve this we could give VADER a rule for *but*, or train Naive Bayes on many more review
sentences instead of 50 general tweets.

## Task 3: Topic Classification

We compare a simple keyword method with LDA (topic modelling from Lab 6).

In [ ]:
def keyword_topic(text):
    text = text.lower()
    if "movie" in text or "film" in text:
        return "movie"
    if "restaurant" in text or "diner" in text or "food" in text:
        return "restaurant"
    if "book" in text or "novel" in text:
        return "book"
    return "unknown"

sent["kw_topic"] = sent["text"].apply(keyword_topic)
print("keyword accuracy:", (sent["kw_topic"] == sent["topic"]).mean())
sent[["sentence id", "topic", "kw_topic"]]

In [ ]:
from gensim import corpora
from gensim.models import LdaModel
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS

# clean each sentence into a list of useful words
texts = []
for sentence in sent["text"]:
    words = [w for w in simple_preprocess(sentence) if w not in STOPWORDS and len(w) > 2]
    texts.append(words)

dictionary = corpora.Dictionary(texts)
corpus = [dictionary.doc2bow(t) for t in texts]

lda = LdaModel(corpus, num_topics=3, id2word=dictionary, passes=30, random_state=42)
for i in range(3):
    print("topic", i, ":", [w for w, p in lda.show_topic(i)])

# give each sentence the topic with the highest probability
predicted = []
for bow in corpus:
    best = max(lda.get_document_topics(bow), key=lambda x: x[1])[0]
    predicted.append(best)

# decide which topic number is which label by looking at the words above
topic_to_label = {0: "movie", 1: "restaurant", 2: "restaurant"}
sent["lda_topic"] = [topic_to_label[t] for t in predicted]
print("LDA accuracy:", (sent["lda_topic"] == sent["topic"]).mean())

### Results and error analysis

The keyword method gets 0.90. It only misses sentence 6 (*Blauwbrug ... favorite place to eat*)
because that sentence does not contain any of the keywords, so it returns *unknown*.

LDA is worse (0.70). With only 10 short sentences there is not enough text for LDA to find clear
topics, and because there are only 2 book sentences it never makes a separate "book" topic - the
book sentences end up mixed with the others. LDA is really made for large collections of documents,
so this result is expected. A keyword list or a trained classifier would work better on such a
small set.

## Summary

| Task | Method 1 | Method 2 | Best |
|------|----------|----------|------|
| NER | spaCy (entity F1 0.76) | NLTK (entity F1 0.46) | spaCy |
| Sentiment | VADER (acc 0.60) | Naive Bayes (acc 0.50) | VADER |
| Topic | Keyword (acc 0.90) | LDA (acc 0.70) | Keyword |

The test set is very small (10 sentences) so we cannot draw strong conclusions. The simple methods
(keyword, VADER) did surprisingly well, while the methods that need more data (Naive Bayes, LDA)
did worse because we did not have enough training data. With more time and more data we would train
the models on review text instead of general tweets.

## Division of work

- **[name 1]**: NER code (spaCy and NLTK), NER error analysis, NER part of the poster.
- **[name 2]**: sentiment code (VADER and Naive Bayes), made the tweet dataset, sentiment analysis, poster layout.
- **[name 3]**: topic code (keyword and LDA), topic analysis, data section of the poster.
- **[name 4]**: evaluation code and tables, summary and conclusions, poster review.

All four of us worked on the code, the analysis and the poster together.